In [ ]:
from ase import Atoms
from ase.optimize import BFGS
from mace.calculators import mace_mp
import json
import numpy as np

# %%
# Load MACE calculator
calc = mace_mp(
    model='mace-mh-1.model',
    default_dtype='float64',
    device='cpu',
    head='omat_pbe'
)

# Load O₂ and H₂ Reference Energies

with open('mace_simple_validation_results/molecular_references.json') as f:
    mol_refs = json.load(f)

E_O2 = mol_refs['O2']['E_total']
E_H2 = mol_refs['H2']['E_total']

print(f"E(O₂) = {E_O2:.6f} eV")
print(f"E(H₂) = {E_H2:.6f} eV")

angle = 104.5 * np.pi / 180  # radians

h2o = Atoms('OH2',
            positions=[
                [0.0, 0.0, 0.0],  # O
                [0.9572 * np.sin(angle/2), 0.9572 * np.cos(angle/2), 0.0],  # H1
                [-0.9572 * np.sin(angle/2), 0.9572 * np.cos(angle/2), 0.0]  # H2
            ])

h2o.center(vacuum=10.0)
h2o.calc = calc

print(f"Initial O-H bonds: {h2o.get_distance(0,1):.4f} Å, {h2o.get_distance(0,2):.4f} Å")
print(f"Initial angle: {h2o.get_angle(1,0,2):.2f}°")

E_initial = h2o.get_potential_energy()
print(f"Initial energy: {E_initial:.6f} eV")
#Relax H₂O with MACE

opt = BFGS(h2o, trajectory='h2o_relax.traj', logfile='h2o_relax.log')
opt.run(fmax=0.01)

E_final = h2o.get_potential_energy()
d_OH1 = h2o.get_distance(0, 1)
d_OH2 = h2o.get_distance(0, 2)
angle_final = h2o.get_angle(1, 0, 2)

print(f"  Final O-H bonds: {d_OH1:.4f} Å, {d_OH2:.4f} Å")
print(f"  Final angle: {angle_final:.2f}°")
print(f"  Final energy: {E_final:.6f} eV")
print(f"  Energy lowering: {E_final - E_initial:.6f} eV")
print(f"  Steps: {opt.get_number_of_steps()}")

print(f"  ΔO-H: {d_OH1 - 0.9572:+.4f} Å ({(d_OH1/0.9572 - 1)*100:+.1f}%)")
print(f"  Δangle: {angle_final - 104.5:+.2f}°")

# Calculate Formation Enthalpy
# H₂ + ½O₂ → H₂O
DH_H2O_mace = E_final - E_H2 - 0.5*E_O2

print(f"  E(H₂O) = {E_final:.6f} eV")
print(f"  E(H₂)  = {E_H2:.6f} eV")
print(f"  E(½O₂) = {0.5*E_O2:.6f} eV")
print(f"\n  ΔHf = {E_final:.6f} - {E_H2:.6f} - {0.5*E_O2:.6f}")
print(f"      = {DH_H2O_mace:.3f} eV")

# Experimental (NIST-JANAF, gas phase, 298 K)
DH_H2O_exp = -2.506  # eV

print(f"  MACE:       {DH_H2O_mace:.3f} eV")
print(f"  Experiment: {DH_H2O_exp:.3f} eV")
print(f"  Error:      {DH_H2O_mace - DH_H2O_exp:+.3f} eV")
print(f"  Error (%):  {(DH_H2O_mace/DH_H2O_exp - 1)*100:+.1f}%")

# Error Decomposition
# Experimental bond energies
E_O2_exp = -5.12  # eV
E_H2_exp = -4.52  # eV

error_O2 = E_O2 - E_O2_exp
error_H2 = E_H2 - E_H2_exp

expected_error = error_H2 + 0.5*error_O2
actual_error = DH_H2O_mace - DH_H2O_exp


# Summary


results = {
    'H2O_geometry': {
        'O-H_bond_initial': 0.9572,
        'O-H_bond_final': d_OH1,
        'angle_initial': 104.5,
        'angle_final': angle_final
    },
    'energies': {
        'E_H2O': E_final,
        'E_H2': E_H2,
        'E_O2': E_O2
    },
    'formation_enthalpy': {
        'DH_mace': DH_H2O_mace,
        'DH_exp': DH_H2O_exp,
        'error_eV': DH_H2O_mace - DH_H2O_exp,
        'error_percent': (DH_H2O_mace/DH_H2O_exp - 1)*100
   
    }
}

# Save
with open('h2o_formation_test.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to: h2o_formation_test.json")
print("\nKey findings:")
print(f"  • MACE ΔHf(H₂O) = {DH_H2O_mace:.3f} eV")
print(f"  • Exp ΔHf(H₂O) = {DH_H2O_exp:.3f} eV")
print(f"  • Error = {DH_H2O_mace - DH_H2O_exp:+.3f} eV ({(DH_H2O_mace/DH_H2O_exp - 1)*100:+.1f}%)")
